# KAN Layers vs Standard PyTorch Layers

This notebook compares KAN implementations with standard PyTorch operations:
- Multiplication: `kan_multiply` vs `*`
- Division: `DivisionKAN` vs `/`
- Softmax: `SoftmaxKAN` vs `F.softmax`
- Attention: `AttentionKAN` vs `nn.MultiheadAttention`

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

from converted_KAN.softmaxkan import (
    kan_multiply, 
    kan_division, 
    kan_reciprocal,
    SoftmaxKAN, 
    DivisionKAN
)
from converted_KAN.attentionkan import (
    kan_matmul,
    AttentionKAN,
    SelfAttentionKAN
)
from converted_KAN import LinearKAN, Conv2dKAN, AvgPool2dKAN, ReLUMaxPool2dKAN
from converted_KAN.ops_counter import count_ops

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Multiplication: `kan_multiply` vs `*`

In [ ]:
# Test multiplication
a = torch.randn(100, 100)
b = torch.randn(100, 100)

# Standard multiplication
result_std = a * b

# KAN multiplication: 4ab = (a+b)² - (a-b)²
result_kan = kan_multiply(a, b)

# Compare
diff = (result_std - result_kan).abs()
print(f"Multiplication Comparison:")
print(f"  Max difference: {diff.max().item():.2e}")
print(f"  Mean difference: {diff.mean().item():.2e}")
print(f"  Numerically equal: {torch.allclose(result_std, result_kan, atol=1e-6)}")

In [ ]:
# Visualize multiplication
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

im0 = axes[0].imshow(result_std[:20, :20].numpy(), cmap='viridis')
axes[0].set_title('Standard a * b')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(result_kan[:20, :20].numpy(), cmap='viridis')
axes[1].set_title('KAN multiply')
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(diff[:20, :20].numpy(), cmap='hot')
axes[2].set_title('Absolute Difference')
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

## 2. Division: `DivisionKAN` vs `/`

In [ ]:
# Test division
a = torch.randn(100, 100)
b = torch.rand(100, 100) + 0.5  # positive values to avoid division issues

# Standard division
result_std = a / b

# KAN division: a/b = a * (1/b)
result_kan = kan_division(a, b)

# Compare
diff = (result_std - result_kan).abs()
print(f"Division Comparison:")
print(f"  Max difference: {diff.max().item():.2e}")
print(f"  Mean difference: {diff.mean().item():.2e}")
print(f"  Numerically equal: {torch.allclose(result_std, result_kan, atol=1e-5)}")

In [ ]:
# Visualize division
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

im0 = axes[0].imshow(result_std[:20, :20].numpy(), cmap='viridis')
axes[0].set_title('Standard a / b')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(result_kan[:20, :20].numpy(), cmap='viridis')
axes[1].set_title('KAN division')
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(diff[:20, :20].numpy(), cmap='hot')
axes[2].set_title('Absolute Difference')
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

## 3. Softmax: `SoftmaxKAN` vs `F.softmax`

In [ ]:
# Test softmax
x = torch.randn(32, 100)  # batch of 32, 100 classes

# Standard softmax
result_std = F.softmax(x, dim=-1)

# KAN softmax
softmax_kan = SoftmaxKAN(dim=-1)
result_kan = softmax_kan(x)

# Compare
diff = (result_std - result_kan).abs()
print(f"Softmax Comparison:")
print(f"  Max difference: {diff.max().item():.2e}")
print(f"  Mean difference: {diff.mean().item():.2e}")
print(f"  Numerically equal: {torch.allclose(result_std, result_kan, atol=1e-6)}")

# Check that outputs sum to 1
print(f"\nSum check (should be 1):")
print(f"  Standard: {result_std.sum(dim=-1).mean().item():.6f}")
print(f"  KAN: {result_kan.sum(dim=-1).mean().item():.6f}")

In [ ]:
# Visualize softmax outputs
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Single sample comparison
sample_idx = 0
axes[0, 0].bar(range(20), result_std[sample_idx, :20].numpy(), alpha=0.7, label='Standard')
axes[0, 0].bar(range(20), result_kan[sample_idx, :20].numpy(), alpha=0.7, label='KAN')
axes[0, 0].set_title('Softmax Output (first 20 classes)')
axes[0, 0].set_xlabel('Class')
axes[0, 0].set_ylabel('Probability')
axes[0, 0].legend()

# Difference histogram
axes[0, 1].hist(diff.flatten().numpy(), bins=50, edgecolor='black')
axes[0, 1].set_title('Distribution of Differences')
axes[0, 1].set_xlabel('Absolute Difference')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_yscale('log')

# Heatmap comparison
im0 = axes[1, 0].imshow(result_std[:10, :30].numpy(), cmap='viridis', aspect='auto')
axes[1, 0].set_title('Standard Softmax')
axes[1, 0].set_xlabel('Class')
axes[1, 0].set_ylabel('Sample')
plt.colorbar(im0, ax=axes[1, 0])

im1 = axes[1, 1].imshow(result_kan[:10, :30].numpy(), cmap='viridis', aspect='auto')
axes[1, 1].set_title('KAN Softmax')
axes[1, 1].set_xlabel('Class')
axes[1, 1].set_ylabel('Sample')
plt.colorbar(im1, ax=axes[1, 1])

plt.tight_layout()
plt.show()

## 4. Matrix Multiplication: `kan_matmul` vs `@`

In [ ]:
# Test matrix multiplication
A = torch.randn(16, 32)  # (M, K)
B = torch.randn(32, 24)  # (K, N)

# Standard matmul
result_std = A @ B

# KAN matmul
result_kan = kan_matmul(A, B)

# Compare
diff = (result_std - result_kan).abs()
print(f"Matrix Multiplication Comparison:")
print(f"  Shape: {A.shape} @ {B.shape} = {result_std.shape}")
print(f"  Max difference: {diff.max().item():.2e}")
print(f"  Mean difference: {diff.mean().item():.2e}")
print(f"  Numerically equal: {torch.allclose(result_std, result_kan, atol=1e-5)}")

In [ ]:
# Visualize matmul
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

im0 = axes[0].imshow(result_std.numpy(), cmap='viridis', aspect='auto')
axes[0].set_title('Standard A @ B')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(result_kan.numpy(), cmap='viridis', aspect='auto')
axes[1].set_title('KAN matmul')
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(diff.numpy(), cmap='hot', aspect='auto')
axes[2].set_title('Absolute Difference')
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

## 5. Attention: `AttentionKAN` vs `nn.MultiheadAttention`

In [ ]:
# Setup attention layers
embed_dim = 64
num_heads = 4
seq_len = 16
batch_size = 2

# Create layers
attn_kan = AttentionKAN(embed_dim, num_heads, dropout=0.0)
attn_std = nn.MultiheadAttention(embed_dim, num_heads, dropout=0.0, batch_first=True)

# Copy weights from KAN to standard for fair comparison
with torch.no_grad():
    # MultiheadAttention uses combined in_proj_weight
    attn_std.in_proj_weight.copy_(torch.cat([
        attn_kan.q_proj.weight,
        attn_kan.k_proj.weight,
        attn_kan.v_proj.weight
    ], dim=0))
    attn_std.in_proj_bias.copy_(torch.cat([
        attn_kan.q_proj.bias,
        attn_kan.k_proj.bias,
        attn_kan.v_proj.bias
    ], dim=0))
    attn_std.out_proj.weight.copy_(attn_kan.out_proj.weight)
    attn_std.out_proj.bias.copy_(attn_kan.out_proj.bias)

print(f"Attention Setup:")
print(f"  Embed dim: {embed_dim}")
print(f"  Num heads: {num_heads}")
print(f"  Head dim: {embed_dim // num_heads}")
print(f"  Sequence length: {seq_len}")
print(f"  Batch size: {batch_size}")

In [ ]:
# Test attention
x = torch.randn(batch_size, seq_len, embed_dim)

# Eval mode to disable dropout
attn_kan.eval()
attn_std.eval()

with torch.no_grad():
    # KAN attention
    result_kan = attn_kan(x, x, x)
    
    # Standard attention
    result_std, _ = attn_std(x, x, x)

# Compare
diff = (result_std - result_kan).abs()
print(f"Attention Comparison:")
print(f"  Output shape: {result_kan.shape}")
print(f"  Max difference: {diff.max().item():.2e}")
print(f"  Mean difference: {diff.mean().item():.2e}")
print(f"  Numerically equal (atol=1e-4): {torch.allclose(result_std, result_kan, atol=1e-4)}")

In [ ]:
# Visualize attention outputs
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# First batch item
for i, (title, result) in enumerate([('Standard', result_std), ('KAN', result_kan)]):
    im = axes[0, i].imshow(result[0].numpy(), cmap='viridis', aspect='auto')
    axes[0, i].set_title(f'{title} Attention Output')
    axes[0, i].set_xlabel('Embedding Dim')
    axes[0, i].set_ylabel('Sequence Position')
    plt.colorbar(im, ax=axes[0, i])

# Difference
im = axes[0, 2].imshow(diff[0].numpy(), cmap='hot', aspect='auto')
axes[0, 2].set_title('Absolute Difference')
axes[0, 2].set_xlabel('Embedding Dim')
axes[0, 2].set_ylabel('Sequence Position')
plt.colorbar(im, ax=axes[0, 2])

# Single position comparison
pos = 0
axes[1, 0].plot(result_std[0, pos].numpy(), label='Standard', alpha=0.7)
axes[1, 0].plot(result_kan[0, pos].numpy(), label='KAN', alpha=0.7, linestyle='--')
axes[1, 0].set_title(f'Output at Position {pos}')
axes[1, 0].set_xlabel('Embedding Dim')
axes[1, 0].set_ylabel('Value')
axes[1, 0].legend()

# Difference histogram
axes[1, 1].hist(diff.flatten().numpy(), bins=50, edgecolor='black')
axes[1, 1].set_title('Distribution of Differences')
axes[1, 1].set_xlabel('Absolute Difference')
axes[1, 1].set_ylabel('Count')

# Scatter plot
axes[1, 2].scatter(result_std.flatten().numpy(), result_kan.flatten().numpy(), alpha=0.3, s=5)
axes[1, 2].plot([-2, 2], [-2, 2], 'r--', label='y=x')
axes[1, 2].set_title('Standard vs KAN')
axes[1, 2].set_xlabel('Standard Output')
axes[1, 2].set_ylabel('KAN Output')
axes[1, 2].legend()

plt.tight_layout()
plt.show()

## 6. Ops Counter (per-layer, ohne Netz)

In [ ]:
def compare_layer_ops(name, std_layer, kan_layer, input_shape):
    std_counts = count_ops(std_layer, input_shape, per_layer=True)
    print(f"{name}:")
    print("  Non-KAN total:", std_counts["total"])
    print("  Non-KAN per-layer:", std_counts["per_layer"])
    try:
        kan_counts = count_ops(kan_layer, input_shape, per_layer=True)
        print("  KAN total:", kan_counts["total"])
        print("  KAN per-layer:", kan_counts["per_layer"])
    except Exception as exc:
        print("  KAN: nicht per fx tracebar:", exc)
    print("")

# Linear
compare_layer_ops(
    "Linear",
    nn.Linear(128, 64),
    LinearKAN(128, 64),
    (1, 128),
)

# Conv2d
compare_layer_ops(
    "Conv2d",
    nn.Conv2d(3, 8, kernel_size=3, padding=1),
    Conv2dKAN(3, 8, kernel_size=3, padding=1),
    (1, 3, 32, 32),
)

# AvgPool2d
compare_layer_ops(
    "AvgPool2d",
    nn.AvgPool2d(kernel_size=2, stride=2),
    AvgPool2dKAN(kernel_size=2, stride=2),
    (1, 8, 32, 32),
)

# MaxPool2d
compare_layer_ops(
    "MaxPool2d",
    nn.MaxPool2d(kernel_size=2, stride=2),
    ReLUMaxPool2dKAN(kernel_size=2, stride=2),
    (1, 8, 32, 32),
)

# Softmax
compare_layer_ops(
    "Softmax",
    nn.Softmax(dim=-1),
    SoftmaxKAN(dim=-1),
    (1, 100),
)


## 7. Performance Comparison

In [ ]:
def benchmark(fn, *args, n_runs=100, warmup=10):
    """Benchmark a function."""
    # Warmup
    for _ in range(warmup):
        fn(*args)
    
    # Benchmark
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    start = time.perf_counter()
    for _ in range(n_runs):
        fn(*args)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    end = time.perf_counter()
    return (end - start) / n_runs * 1000  # ms

In [ ]:
# Benchmark multiplication
sizes = [32, 64, 128, 256, 512]
mul_std_times = []
mul_kan_times = []

for size in sizes:
    a = torch.randn(size, size)
    b = torch.randn(size, size)
    
    mul_std_times.append(benchmark(lambda: a * b))
    mul_kan_times.append(benchmark(lambda: kan_multiply(a, b)))

print("Multiplication Benchmark (ms):")
print(f"{'Size':<10} {'Standard':<15} {'KAN':<15} {'Ratio':<10}")
print("-" * 50)
for i, size in enumerate(sizes):
    ratio = mul_kan_times[i] / mul_std_times[i]
    print(f"{size:<10} {mul_std_times[i]:<15.4f} {mul_kan_times[i]:<15.4f} {ratio:<10.2f}x")

In [ ]:
# Benchmark softmax
softmax_std_times = []
softmax_kan_times = []
softmax_kan = SoftmaxKAN(dim=-1)

for size in sizes:
    x = torch.randn(32, size)
    
    softmax_std_times.append(benchmark(lambda: F.softmax(x, dim=-1)))
    softmax_kan_times.append(benchmark(lambda: softmax_kan(x)))

print("\nSoftmax Benchmark (ms):")
print(f"{'Size':<10} {'Standard':<15} {'KAN':<15} {'Ratio':<10}")
print("-" * 50)
for i, size in enumerate(sizes):
    ratio = softmax_kan_times[i] / softmax_std_times[i]
    print(f"{size:<10} {softmax_std_times[i]:<15.4f} {softmax_kan_times[i]:<15.4f} {ratio:<10.2f}x")

In [ ]:
# Visualize performance
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Multiplication
x_pos = np.arange(len(sizes))
width = 0.35

axes[0].bar(x_pos - width/2, mul_std_times, width, label='Standard', color='steelblue')
axes[0].bar(x_pos + width/2, mul_kan_times, width, label='KAN', color='coral')
axes[0].set_xlabel('Matrix Size')
axes[0].set_ylabel('Time (ms)')
axes[0].set_title('Multiplication Performance')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(sizes)
axes[0].legend()
axes[0].set_yscale('log')

# Softmax
axes[1].bar(x_pos - width/2, softmax_std_times, width, label='Standard', color='steelblue')
axes[1].bar(x_pos + width/2, softmax_kan_times, width, label='KAN', color='coral')
axes[1].set_xlabel('Vector Size')
axes[1].set_ylabel('Time (ms)')
axes[1].set_title('Softmax Performance')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(sizes)
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Summary

In [ ]:
print("=" * 60)
print("SUMMARY: KAN vs Standard Layers")
print("=" * 60)
print()
print("Numerical Accuracy:")
print("  - All KAN implementations are numerically equivalent")
print("  - Small differences due to floating-point arithmetic")
print()
print("KAN Operations (only additions + unary functions):")
print("  - Multiplication: 4ab = (a+b)² - (a-b)²")
print("  - Division: a/b = a × (1/b) with reciprocal")
print("  - Softmax: exp(x) / Σexp(x) via KAN division")
print("  - Attention: Q@K^T via KAN matmul, then softmax, then @V")
print()
print("Performance:")
print("  - KAN operations are slower (more operations)")
print("  - But: Mathematically equivalent to standard ops")
print("  - Theoretical value: Shows that all NNs can be expressed as KANs")